# ESPIRiT on fastMRI brain

The stored brain maps are Walsh. This picks ESPIRiT hyperparameters for
`physics.smaps.espirit` by sweeping them on one slice and scoring two things:

- **residual** `||c - S x|| / ||c||`, where `c = ifft(kspace)` and
  `x = sum_j conj(S_j) c_j`. How well the maps explain the coil images.
- **support** the fraction of pixels with `|S| > 0`. Walsh has no zeros, so
  the organ mask (`smaps.abs().sum(0) > 0`) is all-True with it. ESPIRiT's
  eigenvalue threshold is what puts hard zeros back.

In [ ]:
import os, time, torch, matplotlib.pyplot as plt
os.chdir('/scratch/ee2178/ImMAP')

from datasets.fastmri.common import load_brain_data
from operators.fourier import ifftc
from physics.smaps import espirit

device = 'cuda'
kspace, walsh, _, gnd = load_brain_data(slice_idx=4, device=device)
coils = ifftc(kspace)
print(kspace.shape, kspace.dtype)

In [ ]:
def score(S):
    x = (S.conj() * coils).sum(1, keepdim=True)
    res = ((coils - S * x).norm() / coils.norm()).item()
    sup = (S.abs().sum(1) > 0).float().mean().item()
    return res, sup

print(f"walsh (stored)            residual {score(walsh)[0]:.4f}  support {score(walsh)[1]:.1%}")
print()
print(f"{'acs':>8s} {'ksize':>6s} {'residual':>9s} {'support':>8s} {'s':>6s}")
for acs in (16, 24, 32, 48):
    for ks in (6, 8):
        t = time.time()
        S = espirit(kspace, acs_size=(acs, acs), kernel_size=ks)
        r, u = score(S)
        print(f"{acs:>8d} {ks:>6d} {r:>9.4f} {u:>8.1%} {time.time()-t:>6.1f}")

In [ ]:
# thresh_eig sets the support (hard zeros); thresh_rowspace sets the subspace rank
best = dict(acs_size=(24, 24), kernel_size=6)      # <- edit from the table above

print(f"{'thresh_eig':>11s} {'residual':>9s} {'support':>8s}")
for te in (0.80, 0.90, 0.95, 0.98):
    S = espirit(kspace, thresh_eig=te, **best)
    r, u = score(S)
    print(f"{te:>11.2f} {r:>9.4f} {u:>8.1%}")

In [ ]:
S = espirit(kspace, thresh_eig=0.95, **best)
x = (S.conj() * coils).sum(1, keepdim=True)

fig, ax = plt.subplots(1, 4, figsize=(16, 4))
ax[0].imshow(gnd.abs().squeeze().cpu(), cmap='gray');            ax[0].set_title('ground truth')
ax[1].imshow(x.abs().squeeze().cpu(), cmap='gray');              ax[1].set_title('ESPIRiT combine')
ax[2].imshow(S[0, 0].abs().cpu(), cmap='gray');                  ax[2].set_title('|S| coil 0')
ax[3].imshow((S.abs().sum(1) > 0).squeeze().cpu(), cmap='gray'); ax[3].set_title('support')
for a in ax: a.axis('off')
plt.tight_layout(); plt.show()